# Results Comparison

This notebook will evaluate and compare the performance of all of the models using Tensorboard and code-based metrics.


## Import Dependencies


In [1]:
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

In [2]:
%matplotlib inline

## run tensorboard logging


In [3]:
# %load_ext tensorboard
# %tensorboard --logdir TensorBoard/runs --port 6006

## Read Events Files


### paths and defns


In [4]:
RUNS_DIR: Path = Path('TensorBoard/runs')


MODEL_GROUPS = {
    'YOLO': [
        'yolov8-amy',
        'yolov8-Joachim',
        'yolov11',
        'yolov12'
    ],
    'COCO': [
        'efficieentDet',
        'faster-rcnn',
        'rf-detr',
        'retina-net'
    ],
    'SIGN_ATTR': [
        'sign-condition',
        'sign-shape',
        'viewing-angle',
        'mounting-type'
    ]
}

In [5]:
# fn to load a single file
def extract_scalars_from_event(event_path, model_name, category) -> pd.DataFrame:
    """
    Read a single event file and returns a DataFrame of all scalar metrics.
    """
    # Load the event file with full size guidance
    ea = EventAccumulator(str(event_path), size_guidance={'scalars': 0})
    ea.Reload()
    
    scalar_tags = ea.Tags()['scalars']
    if not scalar_tags:
        return pd.DataFrame() # Return empty if no scalars found

    dfs = []
    
    # Extract data for every metric (tag) found in the file
    for tag in scalar_tags:
        events = ea.Scalars(tag)
        temp_df = pd.DataFrame(events)
        
        # Keep relevant columns and rename
        temp_df = temp_df[['step', 'value']]
        temp_df['metric'] = tag
        dfs.append(temp_df)
        
    # Combine all metrics for this model
    if dfs:
        full_df = pd.concat(dfs, ignore_index=True)
        full_df['model'] = model_name
        full_df['category'] = category
        return full_df
    return pd.DataFrame()

In [6]:
# seqrch for logs

all_dfs = []

for cat, models_list in MODEL_GROUPS.items():
    for model in models_list:
        
        # get model dir
        model_dir = RUNS_DIR / model
        
        # find event files
        event_files = list(model_dir.glob('events.out.tfevents.*'))
        
        while event_files:
            
            # get the latest event file
            latest_event_file = max(event_files, key=os.path.getmtime)
            
            # try save the results
            try:
                df = extract_scalars_from_event(latest_event_file, model, cat)
                if not df.empty:
                    all_dfs.append(df)
                    break  # exit while loop if successful
                else:
                    raise ValueError(f'No scalar data found in event file for model: {model}')
            except Exception as e:
                # catch error, remove current file and try next latest
                print(f'Error processing model: {model}. Error: {e}')
                event_files.remove(latest_event_file)

In [7]:
if all_dfs:
    final_df = pd.concat(all_dfs, ignore_index=True)
    
    print("\nSUCCESS! Data compilation complete.")
    print(f"Total rows: {len(final_df)}")
    print(final_df)
    
    # Optional: Save to CSV for easy access later
    # final_df.to_csv("all_experiment_data.csv", index=False)
else:
    print("No data was extracted. Check your directory paths.")


SUCCESS! Data compilation complete.
Total rows: 30227
       step     value          metric          model   category
0        78  0.680090  train/box_loss     yolov8-amy       YOLO
1        79  0.626840  train/box_loss     yolov8-amy       YOLO
2        80  0.659140  train/box_loss     yolov8-amy       YOLO
3        81  0.569970  train/box_loss     yolov8-amy       YOLO
4        82  0.641400  train/box_loss     yolov8-amy       YOLO
...     ...       ...             ...            ...        ...
30222    26  0.000250          lr/pg7  viewing-angle  SIGN_ATTR
30223    27  0.000203          lr/pg7  viewing-angle  SIGN_ATTR
30224    28  0.000156          lr/pg7  viewing-angle  SIGN_ATTR
30225    29  0.000109          lr/pg7  viewing-angle  SIGN_ATTR
30226    30  0.000061          lr/pg7  viewing-angle  SIGN_ATTR

[30227 rows x 5 columns]
